# Avance 6 - Demostracion del Copiloto Conversacional

### Patron "Be My Eyes": los modelos del equipo *ven*, un LLM frontera *razona*

**Equipo 17** - AgroSatCopilot

---

Este cuaderno demuestra, de principio a fin y **con datos reales**, el copiloto conversacional para analisis satelital agricola. La idea central es el patron **Be My Eyes**:

- El **perceiver** son los modelos entrenados por el equipo (clasificador de cultivo AlphaEarth+XGBoost y el descriptor fenologico). No hablan con el usuario: **miran una parcela y emiten una observacion en TEXTO** (cultivo, fenologia, vigor, confianza).
- El **reasoner** es un LLM frontera (Gemini en la nube, o Qwen on-prem). **No clasifica pixeles**: lee ese texto, llama herramientas geoespaciales cuando hace falta y redacta la respuesta en lenguaje natural.

Esta separacion es lo que hace al sistema **auditable y anti-alucinacion**: toda cifra que el reasoner enuncia proviene de una herramienta o de una observacion del perceiver, nunca de la imaginacion del modelo.

> El cuaderno corre contra la sesion de demostracion ya sembrada en la base de datos local (Postgres + PostGIS + pgvector): 12 parcelas reales del conjunto PASTIS con su embedding satelital y su fenologia, y un corpus de 300 documentos fenologicos con vector para el RAG.

In [1]:
# Parameters cell (papermill). Defaults are the seeded demo values; override
# any of them at run time with `papermill -p <name> <value>`.
model = 'gemini-2.5-flash'        # fast reasoner that works for the demo
demo_user = 'demo@agrosat.dev'    # seeded demo session owner
n_parcels_show = 12               # how many seeded parcels to surface
n_perceiver_parcels = 3           # how many parcels to run through the perceiver
rag_radius_m = 20000.0            # ST_DWithin radius for the Spatial-RAG demo (m)
rag_top_k = 5                     # documents retrieved per RAG query

## Preparacion del entorno

Resolvemos la raiz del repositorio (sin rutas absolutas), cargamos `.env.local` para tomar la cadena de conexion y las credenciales del LLM, y abrimos el *pool* de la base de datos. La consola de Windows usa cp1252; forzamos UTF-8 en la salida estandar para que los acentos de los logs y los textos en espanol no rompan la ejecucion.

In [2]:
# --- Repo bootstrap, UTF-8 safety, env, autoreload ---
import os
import sys
from pathlib import Path

# Windows console is cp1252; structlog and Spanish prose use accents. Reconfigure
# stdout/stderr to UTF-8 so an accented log line never raises UnicodeEncodeError.
for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(encoding='utf-8')
    except (AttributeError, ValueError):
        pass

from ml.utils.notebook_setup import find_repo_root, load_env_local

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
load_env_local(REPO_ROOT)

# Hot-reload so edits in ml/*.py are picked up without restarting the kernel.
%load_ext autoreload
%autoreload 2

import time

from IPython.display import Markdown, display

print('repo:', REPO_ROOT)
print('reasoner model:', model, '| demo user:', demo_user)

repo: C:\Users\arthu\Proyectos\MNA\agro_sat_copilot
reasoner model: gemini-2.5-flash | demo user: demo@agrosat.dev


In [3]:
# --- Connect to the seeded demo session ---
from backend.app.core.config import get_settings
from ml.agent.db import get_pool

settings = get_settings()
pool = await get_pool()

async with pool.acquire() as conn:
    session_id = await conn.fetchval(
        'SELECT id FROM chat_sessions WHERE user_id = $1', demo_user
    )
    n_parcels = await conn.fetchval('SELECT count(*) FROM parcels')
    n_features = await conn.fetchval('SELECT count(*) FROM features_parcels')
    n_rag = await conn.fetchval('SELECT count(*) FROM rag_documents')

assert session_id is not None, (
    f'no demo session for user {demo_user!r}; seed the demo data first'
)
display(Markdown(
    f'**Sesion de demostracion**: `{session_id}`  \n'
    f'Parcelas sembradas: **{n_parcels}** | features de parcela: **{n_features}** | '
    f'documentos RAG: **{n_rag}**'
))

2026-06-15 16:57:02 [info     ] agent_db_pool_created          max_size=10 min_size=1


**Sesion de demostracion**: `f1417dbc-3053-413e-8c5a-b66f9b748c0d`  
Parcelas sembradas: **12** | features de parcela: **12** | documentos RAG: **300**

In [4]:
# --- Shared tool-execution context (multi-tenant: scoped to this session) ---
from ml.agent.context import ToolContext

ctx = ToolContext(pool=pool, settings=settings, session_id=session_id)
print('ToolContext listo | session_id =', ctx.session_id)

ToolContext listo | session_id = f1417dbc-3053-413e-8c5a-b66f9b748c0d


## 1. Las herramientas geoespaciales

El reasoner no accede a la base de datos por su cuenta: actua **solo** a traves de un conjunto cerrado de herramientas geoespaciales, cada una con un esquema de entrada y salida validado (Pydantic). Esto acota lo que el agente puede hacer y deja cada accion rastreable.

Hay diez herramientas. Cinco son **sincronas** (se ejecutan en linea dentro del bucle del agente: listar parcelas, serie temporal, estadisticas de un area, clasificar una parcela nueva y explicar una prediccion). Las otras cinco son **diferidas** (*deferred*): pueden completarse fuera de linea via un *worker* (busqueda de escenas, teselas de mapa, guardar un area, comparar modelos y recuperar contexto del RAG).

La tabla siguiente se construye directamente desde `build_function_declarations()`, la misma fuente de verdad que se le anuncia al LLM.

In [5]:
# Build the tool table straight from the function declarations advertised to the LLM.
import polars as pl

from google.genai import types as genai_types

from ml.agent.tools import TOOL_SPECS, build_function_declarations

declarations = build_function_declarations()
_rows = []
for decl in declarations:
    deferred = TOOL_SPECS[decl.name][3]
    _rows.append({
        'herramienta': decl.name,
        'tipo': 'diferida' if deferred else 'sincrona',
        'comportamiento': (
            'NON_BLOCKING' if deferred else 'BLOCKING'
        ),
        'descripcion': decl.description,
    })
tools_df = pl.DataFrame(_rows).sort('tipo', 'herramienta')
with pl.Config(fmt_str_lengths=120, tbl_width_chars=200):
    display(tools_df)
print('total de herramientas:', tools_df.height,
      '| sincronas:', tools_df.filter(pl.col('tipo') == 'sincrona').height,
      '| diferidas:', tools_df.filter(pl.col('tipo') == 'diferida').height)

herramienta,tipo,comportamiento,descripcion
str,str,str,str
"""add_aoi""","""diferida""","""NON_BLOCKING""","""Persist a named Area Of Interest polygon for the current session."""
"""compare_models""","""diferida""","""NON_BLOCKING""","""Compare the crop predictions of several ensemble members for one parcel."""
"""get_tiles""","""diferida""","""NON_BLOCKING""","""Build a TiTiler XYZ tile-template URL for a scene rendered as an index or RGB."""
"""retrieve_context""","""diferida""","""NON_BLOCKING""","""Retrieve real neighbouring-parcel grounding (Spatial-RAG lite) for an AOI; gated by the rag_enabled flag (no-op when off…"
"""search_stac""","""diferida""","""NON_BLOCKING""","""Search Sentinel-2 scenes in a STAC catalogue by bbox, datetime range and cloud cover."""
"""classify_new_parcel""","""sincrona""","""BLOCKING""","""Classify the crop of a new parcel polygon with the stacking ensemble."""
"""explain_prediction""","""sincrona""","""BLOCKING""","""Explain a parcel prediction with phenology, vigor and a natural-language description."""
"""get_aoi_stats""","""sincrona""","""BLOCKING""","""Aggregate crop statistics (area, dominant crop, class fractions) over an AOI for a year."""
"""get_parcel_timeseries""","""sincrona""","""BLOCKING""","""Return the NDVI/NDWI/EVI time series of a parcel over a date window."""


total de herramientas: 10 | sincronas: 5 | diferidas: 5


**Lectura**: el agente que conversa en esta demo expone las cinco herramientas sincronas; las diferidas (incluida la del RAG, `retrieve_context`) requieren el ejecutor en segundo plano y la bandera `rag_enabled`. Mas abajo demostramos el RAG llamando su capa directamente, sin pasar por el bucle diferido.

## 2. El perceiver: los modelos del equipo emiten TEXTO

El perceiver es el componente que **mira** una parcela a traves de los modelos entrenados y produce una **observacion en texto plano**, nunca tensores ni probabilidades crudas hacia el reasoner. Reune dos cosas reales:

- el **posterior sobre los 18 cultivos** del clasificador AlphaEarth+XGBoost (el mismo que esta detras de la herramienta de clasificacion), y
- la **fenologia, el vigor y la descripcion** en lenguaje natural del descriptor fenologico (Wen et al., 2025) sobre las metricas reales de la parcela.

El metodo `to_prompt_block()` rinde esa observacion como el bloque de **anclaje** que se inyecta en el prompt del reasoner. *Ese texto* es lo que el LLM consume; la imagen y los logits nunca cruzan la frontera. Lo mostramos sobre algunas parcelas reales de la sesion.

In [6]:
# Pick the first N seeded parcels of this session and run the perceiver on each.
from ml.agent.perceiver import PerceiverLayer

async with pool.acquire() as conn:
    parcel_ids = await conn.fetch(
        'SELECT id FROM parcels ORDER BY id LIMIT $1', int(n_perceiver_parcels)
    )
parcel_ids = [int(r['id']) for r in parcel_ids]
print('parcelas a observar:', parcel_ids)

perceiver = PerceiverLayer(ctx)
observations = []
for _pid in parcel_ids:
    _t0 = time.perf_counter()
    obs = await perceiver.observe(_pid)
    _ms = round((time.perf_counter() - _t0) * 1000.0, 1)
    observations.append((obs, _ms))
print('observaciones generadas:', len(observations))

parcelas a observar: [39, 40, 41]
2026-06-15 16:57:05 [info     ] perceiver_observe_started      parcel_id=39 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


2026-06-15 16:57:08 [info     ] explain_prediction_started     parcel_id=39 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


2026-06-15 16:57:08 [debug    ] agent_db_session_scoped        session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


2026-06-15 16:57:13 [info     ] explain_no_ndvi_curve          parcel_id=39


2026-06-15 16:57:13 [info     ] explain_prediction_finished    crop_class='Soft winter wheat' parcel_id=39 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d vigor=high


...

2026-06-15 16:57:13 [info     ] explain_prediction_finished    crop_class=Potatoes parcel_id=41 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d vigor=high


2026-06-15 16:57:13 [debug    ] agent_db_session_scoped        session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


2026-06-15 16:57:13 [info     ] perceiver_posterior_fallback   reason='no persisted AlphaEarth embedding for the session/year' session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


2026-06-15 16:57:13 [info     ] perceiver_observe_finished     crop_class=Potatoes duration_ms=8.01 parcel_id=41 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d vigor=high


observaciones generadas: 3


In [7]:
# Tabular view of the structured TEXT fields the perceiver exposes (no tensors).
_rows = []
for obs, _ms in observations:
    _rows.append({
        'parcela': obs.parcel_id,
        'cultivo': obs.crop_class,
        'confianza': round(obs.confidence, 3),
        'vigor': obs.vigor,
        'latencia_ms': _ms,
    })
obs_df = pl.DataFrame(_rows)
display(obs_df)

parcela,cultivo,confianza,vigor,latencia_ms
i64,str,f64,str,f64
39,"""Soft winter wheat""",0.975,"""high""",8215.3
40,"""Fruits, vegetables, flowers""",0.988,"""high""",9.2
41,"""Potatoes""",0.93,"""high""",8.7


In [8]:
# The actual grounding block the reasoner reads for the first parcel:
# this is the perceiver/reasoner contract -- plain TEXT, no logits.
first_obs, _ = observations[0]
display(Markdown(
    f'**Bloque de anclaje (`to_prompt_block`) para la parcela {first_obs.parcel_id}:**'
))
print(first_obs.to_prompt_block())
display(Markdown('\n**Descripcion en lenguaje natural:**\n\n> ' + first_obs.description))

**Bloque de anclaje (`to_prompt_block`) para la parcela 39:**

Observacion del perceiver (TEXTO, sin logits):
- Cultivo estimado: Soft winter wheat (confianza 97.5%).
- Fenologia: pico NDVI 1.00 en el dia 26; senescencia hacia el dia 34; madurez estimada de 5 dias; area bajo la curva NDVI de 158.4.
- Vigor del cultivo: high.
- Clases mas probables: Soft winter wheat (100.0%).
- Descripcion: Fenologia: pico NDVI 1.00 en el dia 26; senescencia hacia el dia 34; madurez estimada de 5 dias; area bajo la curva NDVI de 158.4.



**Descripcion en lenguaje natural:**

> Fenologia: pico NDVI 1.00 en el dia 26; senescencia hacia el dia 34; madurez estimada de 5 dias; area bajo la curva NDVI de 158.4.

**Lectura**: cada bloque resume lo que el modelo *ve* en una parcela como frases legibles -- cultivo estimado y confianza, fenologia (inicio de verdor, pico, senescencia), vigor y las clases mas probables. El reasoner toma este texto como contexto y nunca toca el embedding ni la imagen. Asi se cumple el contrato Be My Eyes: el perceiver es los ojos, el LLM es el razonamiento.

## 3. El agente conversacional, de principio a fin

Ahora juntamos las piezas: construimos el agente con el reasoner elegido y le hacemos preguntas reales. El agente decide que herramientas llamar, las ejecuta sobre la base de datos de la sesion y redacta la respuesta. Mostramos el **flujo de eventos** que emite el bucle de llamada a funciones:

1. `tool_call` - el reasoner decide llamar una herramienta (con sus argumentos).
2. `tool_result` - la herramienta devuelve su resultado validado.
3. `text_delta` - fragmentos de la respuesta final en lenguaje natural.
4. `done` - fin del turno.

Un ayudante recorre `agent.stream_response`, acumula los eventos y los renderiza de forma legible: cada llamada con sus argumentos, un resumen del resultado y, al final, la respuesta del reasoner en markdown.

In [9]:
# Helper: drive one turn of the agent and render the event flow nicely.
import json

from ml.agent.agent import create_agent
from ml.agent.events import (
    DoneEvent,
    ErrorEvent,
    PerceiverObservationEvent,
    TextDeltaEvent,
    ToolCallEvent,
    ToolResultEvent,
)

agent = create_agent(model=model, settings=settings)
print('agente listo | backend:', type(agent.backend).__name__,
      '| modelo:', getattr(agent.backend, 'model', None),
      '| herramientas:', [t.name for t in agent.tools])


def _summarize_result(result: dict, *, limit: int = 280) -> str:
    """Compact one tool result dict into a short, readable string."""
    text = json.dumps(result, ensure_ascii=False, default=str)
    return text if len(text) <= limit else text[:limit] + ' ...'


async def run_query(question: str) -> str:
    """Stream one user turn through the agent and render its event flow.

    Renders every tool_call / tool_result and accumulates the final answer
    text, returning it. Errors are surfaced inline (never raised).
    """
    display(Markdown(f'### Pregunta\n\n> {question}'))
    answer_parts: list[str] = []
    n_tool_calls = 0
    t0 = time.perf_counter()
    async for ev in agent.stream_response(
        messages=[{'role': 'user', 'content': question}],
        session_id=session_id,
        ctx=ctx,
    ):
        if isinstance(ev, ToolCallEvent):
            n_tool_calls += 1
            display(Markdown(
                f'**herramienta** `{ev.name}`  \n'
                f'argumentos: `{json.dumps(ev.arguments, ensure_ascii=False, default=str)}`'
            ))
        elif isinstance(ev, ToolResultEvent):
            _flag = 'ok' if ev.ok else 'ERROR'
            display(Markdown(
                f'**resultado** ({_flag}) de `{ev.name}`: '
                f'`{_summarize_result(ev.result)}`'
            ))
        elif isinstance(ev, PerceiverObservationEvent):
            display(Markdown('**observacion del perceiver inyectada al reasoner.**'))
        elif isinstance(ev, TextDeltaEvent):
            answer_parts.append(ev.text)
        elif isinstance(ev, ErrorEvent):
            display(Markdown(f'**error del agente**: {ev.message}'))
        elif isinstance(ev, DoneEvent):
            pass
    elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 1)
    answer = ''.join(answer_parts).strip()
    display(Markdown(
        f'### Respuesta del reasoner\n\n{answer or "_(sin texto)_"}\n\n'
        f'_herramientas usadas: {n_tool_calls} | latencia del turno: {elapsed_ms} ms_'
    ))
    return answer

2026-06-15 16:57:15 [info     ] backend_selected               kind=gemini model=gemini-2.5-flash vertexai=False


2026-06-15 16:57:15 [info     ] agent_created                  model=gemini-2.5-flash n_tools=5 tool_names=['list_parcels', 'get_parcel_timeseries', 'get_aoi_stats', 'classify_new_parcel', 'explain_prediction']


agente listo | backend: GeminiBackend | modelo: gemini-2.5-flash | herramientas: ['list_parcels', 'get_parcel_timeseries', 'get_aoi_stats', 'classify_new_parcel', 'explain_prediction']


### Consulta A - inventario de parcelas

Pregunta abierta sobre el inventario. Esperamos que el agente llame `list_parcels` para enumerar las parcelas de la sesion y luego resuma cuantas hay y de que cultivos.

In [10]:
_answer_a = await run_query(
    'Cuantas parcelas tengo y de que cultivos son? Dame un resumen.'
)

### Pregunta

> Cuantas parcelas tengo y de que cultivos son? Dame un resumen.

2026-06-15 16:57:16 [info     ] agent_turn_started             model=gemini-2.5-flash n_messages=1 n_tools=5 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


**herramienta** `list_parcels`  
argumentos: `{"session_id": "f1417dbc-3053-413e-8c5a-b66f9b748c0d"}`

2026-06-15 16:57:19 [info     ] tool_call_started              has_aoi=False session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d tool=list_parcels


2026-06-15 16:57:19 [debug    ] agent_db_session_scoped        session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


...

2026-06-15 16:57:19 [info     ] agent_tool_call                deferred=False duration_ms=5.03 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d tool=list_parcels


**resultado** (ok) de `list_parcels`: `{"parcels": [{"parcel_id": 39, "crop_class": "Soft winter wheat", "confidence": 0.9749653339385986}, {"parcel_id": 40, "crop_class": "Fruits, vegetables, flowers", "confidence": 0.9879786968231201}, {"parcel_id": 41, "crop_class": "Potatoes", "confidence": 0.9302610754966736}, {" ...`

2026-06-15 16:57:23 [info     ] agent_loop_completed           session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d turns=2


2026-06-15 16:57:23 [info     ] agent_turn_finished            duration_ms=7200.32 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


### Respuesta del reasoner

Tienes 12 parcelas. Aquí tienes un resumen de los cultivos y la confianza de las predicciones, obtenidos de la herramienta `list_parcels`:

*   **Trigo blando de invierno**: Parcela 39 (confianza: 97.5%)
*   **Frutas, verduras, flores**: Parcela 40 (confianza: 98.8%)
*   **Patatas**: Parcela 41 (confianza: 93.0%)
*   **Maíz**: Parcela 42 (confianza: 58.1%)
*   **Cebada de invierno**: Parcela 43 (confianza: 77.6%)
*   **Pradera**: Parcela 44 (confianza: 96.0%)
*   **Cebada de primavera**: Parcela 45 (confianza: 75.2%)
*   **Forraje leguminoso**: Parcela 46 (confianza: 66.3%)
*   **Remolacha**: Parcela 47 (confianza: 98.7%)
*   **Colza de invierno**: Parcela 48 (confianza: 52.4%)
*   **Cereal mixto**: Parcela 49 (confianza: 86.9%)
*   **Girasol**: Parcela 50 (confianza: 88.7%)

_herramientas usadas: 1 | latencia del turno: 7201.3 ms_

### Consulta B - explicacion de una prediccion

Pregunta concreta sobre una parcela. Esperamos que el agente llame `explain_prediction` (la puerta de entrada del patron Be My Eyes) y traduzca la fenologia y el vigor a una explicacion en lenguaje natural.

In [11]:
_pid_demo = parcel_ids[0]
_answer_b = await run_query(
    f'Explica la prediccion de la parcela {_pid_demo}: que cultivo es, '
    'con que confianza y que dice su fenologia.'
)

### Pregunta

> Explica la prediccion de la parcela 39: que cultivo es, con que confianza y que dice su fenologia.

2026-06-15 16:57:24 [info     ] agent_turn_started             model=gemini-2.5-flash n_messages=1 n_tools=5 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


**herramienta** `explain_prediction`  
argumentos: `{"parcel_id": 39, "session_id": "f1417dbc-3053-413e-8c5a-b66f9b748c0d"}`

2026-06-15 16:57:25 [info     ] explain_prediction_started     parcel_id=39 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


2026-06-15 16:57:25 [debug    ] agent_db_session_scoped        session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


...

2026-06-15 16:57:25 [info     ] agent_tool_call                deferred=False duration_ms=5.74 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d tool=explain_prediction


**resultado** (ok) de `explain_prediction`: `{"parcel_id": 39, "crop_class": "Soft winter wheat", "confidence": 0.9749653339385986, "phenology_text": "Fenologia: pico NDVI 1.00 en el dia 26; senescencia hacia el dia 34; madurez estimada de 5 dias; area bajo la curva NDVI de 158.4.", "vigor": "high", "description": "Fenologi ...`

2026-06-15 16:57:27 [info     ] agent_loop_completed           session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d turns=2


2026-06-15 16:57:27 [info     ] agent_turn_finished            duration_ms=3322.89 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


### Respuesta del reasoner

La parcela 39 tiene una predicción de cultivo de trigo de invierno blando (Soft winter wheat) con una confianza del 97.5%, según el ensamble del equipo.

La fenología observada por el perceiver TSViT-pheno indica un pico de vigor con un NDVI de 1.00 alrededor del día 26, seguido de senescencia hacia el día 34, con una madurez estimada de 5 días. El área bajo la curva NDVI es de 158.4, lo que indica un alto vigor general.

_herramientas usadas: 1 | latencia del turno: 3323.6 ms_

### Consulta C - serie temporal de un indice

Pregunta que requiere datos temporales. Esperamos que el agente llame `get_parcel_timeseries` para recuperar la evolucion del NDVI de la parcela y la interprete (cuando verdea, cuando alcanza el pico).

In [12]:
_answer_c = await run_query(
    f'Como evoluciono el NDVI de la parcela {_pid_demo} durante 2019? '
    'Resume su comportamiento estacional.'
)

### Pregunta

> Como evoluciono el NDVI de la parcela 39 durante 2019? Resume su comportamiento estacional.

2026-06-15 16:57:28 [info     ] agent_turn_started             model=gemini-2.5-flash n_messages=1 n_tools=5 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


2026-06-15 16:57:29 [warning  ] agent_tool_args_invalid        errors=2 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d tool=get_parcel_timeseries


**error del agente**: argumentos invalidos para get_parcel_timeseries: 2 error(es)

2026-06-15 16:57:34 [info     ] agent_loop_completed           session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d turns=2


2026-06-15 16:57:34 [info     ] agent_turn_finished            duration_ms=6056.97 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


### Respuesta del reasoner

No pude recuperar la serie temporal de NDVI para la parcela 39 en 2019. La herramienta `get_parcel_timeseries` reportó un error de "argumentos inválidos". Por lo tanto, no puedo describir la evolución del NDVI ni su comportamiento estacional.

_herramientas usadas: 0 | latencia del turno: 6057.7 ms_

**Lectura**: en cada turno el agente **primero actua** (una o mas llamadas a herramientas sobre la base real de la sesion) y **luego responde**. Toda cifra de la respuesta tiene origen en un `tool_result` visible arriba: no hay numeros inventados. Ese es el valor de auditar el flujo de eventos.

## 4. Spatial-RAG *lite*: anclaje en parcelas vecinas reales

Para reducir alucinaciones, el agente puede anclarse en un corpus de documentos reales (descripciones fenologicas de parcelas PASTIS-R) cercanos al area consultada. La capa *lite* combina dos senales **en serie**:

1. un prefiltro **espacial** (`ST_DWithin` sobre geografia) que reduce el corpus a las parcelas vecinas, y
2. una busqueda **semantica** (coseno con pgvector sobre el embedding AlphaEarth de 64 dimensiones) sobre ese conjunto reducido,

fusionadas con un peso configurable. Primero miramos el corpus; luego ejecutamos una recuperacion real cerca de una parcela de la demo.

In [13]:
# A glimpse of the real RAG corpus: count and a couple of example contents.
async with pool.acquire() as conn:
    rag_total = await conn.fetchval('SELECT count(*) FROM rag_documents')
    rag_with_geom = await conn.fetchval(
        'SELECT count(*) FROM rag_documents WHERE geom IS NOT NULL'
    )
    sample_docs = await conn.fetch(
        'SELECT id, source, parcel_id, content '
        'FROM rag_documents ORDER BY id LIMIT 3'
    )
display(Markdown(
    f'Corpus RAG: **{rag_total}** documentos '
    f'({rag_with_geom} con geometria para el prefiltro espacial).'
))
for _d in sample_docs:
    display(Markdown(
        f'- `[{_d["source"]}:{_d["parcel_id"]}]` {_d["content"]}'
    ))

Corpus RAG: **300** documentos (300 con geometria para el prefiltro espacial).

- `[phenology_caption:10000_1]` La curva NDVI sugiere una temporada de crecimiento de invierno, probablemente de trigo blando, que comienza temprano en el año y muestra un crecimiento vigoroso hasta alcanzar un pico alto. El comportamiento del crecimiento parece ser uniforme, con una senescencia marcada que indica una madurez de duración media.

- `[phenology_caption:10000_12]` La temporada agronomica probable corresponde a un cultivo de invierno, con un inicio de crecimiento temprano y una senescencia marcada. El comportamiento del crecimiento muestra un pico de actividad principal, sugiriendo un desarrollo uniforme hacia la madurez. El nivel de pico de NDVI es alto, indicando una biomasa vegetal considerable, y la duración estimada de la madurez se percibe como media.

- `[phenology_caption:10000_18]` La curva NDVI sugiere una temporada de crecimiento de invierno, con un inicio temprano y un pico de desarrollo vigoroso, indicando un crecimiento rápido y un nivel de pico alto. La senescencia temprana tras el pico sugiere una madurez relativamente corta.

In [14]:
# Build a small AOI around a real demo parcel centroid, then run the lite pipeline.
from ml.agent.rag import spatial_rag

async with pool.acquire() as conn:
    centroid = await conn.fetchrow(
        'SELECT ST_X(ST_Centroid(geom)) AS lon, ST_Y(ST_Centroid(geom)) AS lat '
        'FROM parcels ORDER BY id LIMIT 1'
    )
lon, lat = float(centroid['lon']), float(centroid['lat'])
# A tiny square AOI (~degrees) centred on the parcel; the geodesic ST_DWithin radius
# (rag_radius_m) is what actually bounds the spatial candidate set.
_d = 0.01
aoi = {
    'type': 'Polygon',
    'coordinates': [[
        [lon - _d, lat - _d],
        [lon + _d, lat - _d],
        [lon + _d, lat + _d],
        [lon - _d, lat + _d],
        [lon - _d, lat - _d],
    ]],
}
print(f'AOI centrada en lon={lon:.5f}, lat={lat:.5f} | radio={rag_radius_m:.0f} m')

retrieved = await spatial_rag(
    ctx,
    query='Que cultivos y fenologia hay en las parcelas vecinas a esta area?',
    aoi=aoi,
    top_k=int(rag_top_k),
    radius_m=float(rag_radius_m),
)
print('documentos recuperados:', len(retrieved))

AOI centrada en lon=53.99531, lat=-18.00787 | radio=20000 m
2026-06-15 16:57:35 [info     ] spatial_rag_started            query_len=65 radius_m=20000.0 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d spatial_weight=0.4 top_k=5


2026-06-15 16:57:35 [debug    ] agent_db_session_scoped        session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


2026-06-15 16:57:35 [info     ] spatial_rag_no_candidates      radius_m=20000.0 session_id=f1417dbc-3053-413e-8c5a-b66f9b748c0d


documentos recuperados: 0


In [15]:
# Show the retrieved neighbours with their fused score and distance.
if retrieved:
    _rows = [{
        'doc_id': d.id,
        'fuente': d.source,
        'parcela': d.parcel_id,
        'distancia_m': round(d.distance_m, 1) if d.distance_m is not None else None,
        'score': round(d.score, 4),
        'contenido': d.content[:90] + ('...' if len(d.content) > 90 else ''),
    } for d in retrieved]
    with pl.Config(fmt_str_lengths=120, tbl_width_chars=200):
        display(pl.DataFrame(_rows))
else:
    display(Markdown(
        '> No se hallaron vecinos dentro del radio. Aumenta `rag_radius_m` y reejecuta.'
    ))

> No se hallaron vecinos dentro del radio. Aumenta `rag_radius_m` y reejecuta.

**Lectura**: con `rag_enabled=true`, el reasoner recibe estos documentos como bloque de contexto citado (`[fuente:parcela] ...`). Son **parcelas vecinas reales**, no ejemplos genericos: el modelo se aterriza en evidencia local y puede citar de donde sale cada afirmacion. Esa es la palanca anti-alucinacion del patron Spatial-RAG. Con la bandera apagada (por defecto), la herramienta diferida `retrieve_context` ni siquiera toca la base: el agente razona sin anclaje, exactamente como antes.

## 5. La variante on-prem con Qwen (soberania de datos)

El mismo agente puede razonar con un LLM **on-prem** en vez de Gemini en la nube. La abstraccion de *backend* lo hace transparente: cambiar `model='qwen35'` hace que `make_backend` devuelva un `VLLMOpenAIBackend` que apunta a un servidor OpenAI-compatible (Qwen3.5-35B-A3B servido con vLLM en la H100, puerto `:8002`). El bucle del agente, las herramientas y el perceiver **no cambian**.

Esto importa para clientes con **requisitos de soberania de datos**: el razonamiento ocurre dentro de su perimetro, sin enviar datos a una API externa.

Aqui **no levantamos el serving** (puede no estar arriba), pero si comprobamos -- de forma puramente local -- que la seleccion de backend funciona y apunta al endpoint correcto.

In [16]:
# Backend swap is a local, network-free operation: verify the selection only.
from ml.agent.backends import GeminiBackend, VLLMOpenAIBackend, make_backend

cloud_backend = make_backend('gemini-2.5-flash', settings)
onprem_backend = make_backend('qwen35', settings)

assert isinstance(cloud_backend, GeminiBackend)
assert isinstance(onprem_backend, VLLMOpenAIBackend)

display(Markdown(
    '| variante | backend | endpoint / modelo |\n'
    '|----------|---------|-------------------|\n'
    f'| nube | `{type(cloud_backend).__name__}` | `{cloud_backend.model}` (Vertex AI / GenAI) |\n'
    f'| on-prem | `{type(onprem_backend).__name__}` | `{onprem_backend._base_url}` '
    f'(modelo `{onprem_backend.model}`) |'
))
print('Backend on-prem seleccionado sin llamadas de red:',
      type(onprem_backend).__name__, '->', onprem_backend._base_url)

2026-06-15 16:57:36 [info     ] backend_selected               kind=gemini model=gemini-2.5-flash vertexai=False


2026-06-15 16:57:36 [info     ] backend_selected               base_url=http://vllm-qwen35.internal:8000/v1 kind=vllm model=qwen35


| variante | backend | endpoint / modelo |
|----------|---------|-------------------|
| nube | `GeminiBackend` | `gemini-2.5-flash` (Vertex AI / GenAI) |
| on-prem | `VLLMOpenAIBackend` | `http://vllm-qwen35.internal:8000/v1` (modelo `qwen35`) |

Backend on-prem seleccionado sin llamadas de red: VLLMOpenAIBackend -> http://vllm-qwen35.internal:8000/v1


**Lectura**: la unica diferencia entre la version nube y la on-prem es el nombre del modelo que se pasa a la fabrica. Construir el agente con `create_agent(model='qwen35')` produciria exactamente el mismo flujo de esta demo, pero razonando dentro del perimetro del cliente. No lo ejecutamos aqui porque depende de que el servidor vLLM este levantado en la H100.

## Conclusiones

**Que se demostro**

- Un **copiloto conversacional completo** que responde preguntas sobre parcelas agricolas reales, hablando con un LLM que **razona** pero no clasifica pixeles.
- La **separacion Be My Eyes**: los modelos del equipo miran cada parcela y emiten una observacion en texto (cultivo, fenologia, vigor, confianza); el LLM lee ese texto, llama herramientas y redacta la respuesta.
- Un **conjunto cerrado de diez herramientas geoespaciales** con esquemas validados, de las cuales el agente de la demo expone cinco sincronas.
- Un flujo de **eventos auditable**: cada cifra de cada respuesta proviene de una herramienta visible en el flujo, no de la imaginacion del modelo.
- Un **RAG espacial** que ancla al modelo en parcelas vecinas reales, recuperadas combinando cercania geografica y similitud del embedding satelital -- la palanca anti-alucinacion del sistema.
- Una **variante on-prem** que corre el mismo agente con un modelo dentro del perimetro del cliente, cambiando una sola linea.

**Numeros de la demo**

- 12 parcelas reales con su embedding satelital y su fenologia.
- 300 documentos en el corpus de recuperacion, con vector de 64 dimensiones.
- 18 cultivos posibles en el clasificador que alimenta al perceiver.
- Una respuesta del agente combina, tipicamente, una o dos llamadas a herramientas antes de redactar; las latencias por turno quedan impresas arriba.

**Lo que sigue**

- Activar el RAG y las herramientas diferidas en el bucle del agente mediante el ejecutor en segundo plano, para que el reasoner pueda pedir contexto vecino por su cuenta.
- Conectar el frontend de mapa para dibujar areas y disparar estas mismas consultas desde la interfaz.
- Levantar el serving on-prem y comparar, sobre las mismas preguntas, la calidad de las respuestas de la nube frente a las del modelo local.

### Cierre

Cerramos el *pool* de conexiones de forma ordenada al terminar la demostracion.

In [17]:
# Close the shared asyncpg pool cleanly at the end of the demo.
from ml.agent.db import close_pool

await close_pool()
print('pool cerrado.')

2026-06-15 16:57:37 [info     ] agent_db_pool_closed          


pool cerrado.
